# 03 Feld & Comps

**Eingabe**: Zielname (SIMBAD), Sensorfeld, optional Kamerawinkel  
**Ausgabe**: `field_config.json` mit Ziel, Comp-Sternen, Check-Stern und Reserve — plus Himmelskarte der Auswahl

Ablauf (ohne Bild, nur Kataloge — Netz nötig, Ergebnisse werden in `~/.cache/varphot` zwischengespeichert):
1. Ziel bei **SIMBAD** auflösen (RA/Dec, B, V → B−V des Ziels)
2. **AAVSO-VSP**-Sequenz für das Feld holen (kalibrierte V/B-Helligkeiten mit Fehlern)
3. Regeln anwenden (Feld, Helligkeit, Fehler, Farbe, Veränderlichkeits-Hinweise) und die Überlebenden gegen **Gaia DR3** prüfen (Nachbarn, Variabilität, RUWE, V-Konsistenz)
4. Karte ansehen, Ergebnis speichern

Die Zuordnung zu den Sternen im **Bild** (Detektion + Crossmatch) folgt, sobald ein Bild des Zielfeldes vorliegt (Roadmap P0-3).

## Schritt 1 — Ziel und Parameter

`ZIEL` ist der SIMBAD-Bezeichner (z. B. `del Cep`, `eta Aql`, `V* RR Lyr`); `VSP_STAR` der Name für die AAVSO-Karte.
`FIELD_DEG` ist das Sensorfeld (Standard: Apo 80/540 + QHY268M = 2,49° × 1,67°). `PA_DEG = None` wählt nur Sterne, die bei **jeder**
Kamerarotation im Bild liegen (Kreis); ist der Winkel bekannt (aus der WCS nach dem Plate Solve), `PA_DEG` setzen — dann gilt das
gedrehte Rechteck und mehr Sterne bleiben im Rennen. `MAGLIMIT` begrenzt die Sequenz (VSP-Abruf bis V ≈ MAGLIMIT).

In [ ]:
import json, datetime
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from varphot import __version__
from varphot.vsp import (resolve_target, fetch_vsp, select_comps_catalog, reject_summary, VSP_CFG)
from varphot import plot_comp_field

ZIEL      = "del Cep"
VSP_STAR  = "Delta Cep"
FIELD_DEG = (2.49, 1.67)     # Sensorfeld [Grad]: lange × kurze Seite
PA_DEG    = None             # None = rotationssicher; sonst Winkel der langen Seite [Grad]
MAGLIMIT  = 11.0
FOV_ARCMIN = 180             # VSP-Kartengröße (max. 180′)

CFG_FELD = {"field_deg": FIELD_DEG, "field_pa_deg": PA_DEG}   # weitere Regeln: VSP_CFG in varphot/vsp.py
OUTPUT = Path("../data/fields") / (ZIEL.replace(" ", "_") + "_field_config.json")
OUTPUT.parent.mkdir(parents=True, exist_ok=True)

target = resolve_target(ZIEL)
print(f"Ziel: {target['name']}  ({target['otype']})")
print(f"  RA {target['ra']:.5f}°  Dec {target['dec']:+.5f}°")
print(f"  V = {target['V']}  B = {target['B']}  B−V = {target['BV']:.2f}")

## Schritt 2 — AAVSO-VSP-Sequenz

Lädt die offizielle AAVSO-Vergleichssternsequenz (VSP-API) für das Zielfeld. Jeder Stern hat eine AUID, Position, V/B(/Rc/Ic) mit Fehlern
und einen Kommentar, der z. B. „NSV" (Verdacht auf Veränderlichkeit) oder „nur visuell" enthalten kann — diese Hinweise werten wir aus.

In [ ]:
vsp = fetch_vsp(VSP_STAR, fov_arcmin=FOV_ARCMIN, maglimit=MAGLIMIT)
print(f"{len(vsp)} Sterne in der AAVSO-Sequenz (V ≤ {MAGLIMIT}, Karte {FOV_ARCMIN}′)")
vsp[["auid", "label", "V", "V_err", "B", "BV", "comment"]][:8]

## Schritt 3 — Regeln und Gaia-Prüfung

`select_comps_catalog()` bewertet **jeden** VSP-Stern und weist eine Rolle zu: `comp` (Ensemble, bis zu 8), `check` (nächstbester,
nicht im Ensemble), `reserve` oder `verworfen`. Jeder verworfene Stern behält seine Begründung(en) — nichts verschwindet stillschweigend.
Stufe 1 braucht nur die VSP-Daten; Stufe 2 (Gaia DR3 über das ESA-TAP) läuft nur für die Überlebenden von Stufe 1.

In [ ]:
res = select_comps_catalog(vsp, target, CFG_FELD)

from collections import Counter
cnt = Counter(map(str, res["role"]))
print("Rollen:", {k: cnt.get(k, 0) for k in ("comp", "check", "reserve", "verworfen")})
print("Ablehnungsgründe (ein Stern kann mehrere haben):")
for k, n in reject_summary(res).items():
    print(f"  {n:3d} × {k}")
if cnt.get("comp", 0) < 5:
    print("\n⚠ Weniger als 5 Comps aus der AAVSO-Sequenz — Schritt 3b ergänzt mit Gaia-Feldsternen; "
          "Sensorwinkel PA_DEG bekannt? Dann gilt das gedrehte Rechteck und mehr VSP-Sterne bleiben übrig.")

cols = ["role", "label", "V", "V_err", "BV", "sep_deg", "G", "V_gaia", "reasons"]
df = res[cols].to_pandas()
df["role"] = pd.Categorical(df["role"], ["comp", "check", "reserve", "verworfen"], ordered=True)
display(df.sort_values(["role", "sep_deg"]).round(3).reset_index(drop=True))

### Schritt 3b — Ensemble mit Gaia-Feldsternen auffüllen (optional)

Liefert die AAVSO-Sequenz zu wenige Comps (Ziel: 5–10 plus Check), ergänzt `fill_with_gaia()` das Ensemble mit **Gaia-DR3-Feldsternen**:
im Feld, V aus G und BP−RP (Polynom, σ ≈ 0,04 mag), Farbe nah am Ziel, kein Gaia-„VARIABLE", RUWE < 1,4, Einzelstern, ohne helle Nachbarn im Aperturbereich
(nacheinander per ESA-TAP geprüft, danach im Cache). Gaia-Sterne stehen in der Karte als **Quadrate** und tragen `source = "Gaia"`; ihr V ist **abgeleitet**, nicht
AAVSO-kalibriert — vor dem Einreichen gegen die VSP-Comps gegenprüfen. `USE_GAIA_FILL = False` schaltet den Schritt ab.

In [ ]:
USE_GAIA_FILL = True

if USE_GAIA_FILL:
    from varphot.vsp import fill_with_gaia
    n_vorher = int((res["role"] == "comp").sum())
    res = fill_with_gaia(res, target, CFG_FELD)
    g = res[[str(s) == "Gaia" for s in res["source"]]]
    print(f"Comps: {n_vorher} (VSP) → {int((res['role'] == 'comp').sum())} (mit Gaia); "
          f"{len(g)} Gaia-Kandidaten geprüft, davon {sum(str(r) != 'verworfen' for r in g['role'])} verwendet")
    display(g[["role", "label", "auid", "G", "bp_rp", "V", "sep_deg", "reasons"]].to_pandas().round(3))
else:
    print("Gaia-Ergänzung aus (USE_GAIA_FILL = False)")

## Schritt 4 — Karte der Auswahl

Himmelskarte um das Ziel (Osten links). Rot = Sensorfeld (Kreis = rotationssicher, Rechteck = bei bekanntem Winkel), grün = Comps,
orange = Check, blau = Reserve, grau × = verworfen (Nummer = AAVSO-Label).

In [ ]:
fig = plot_comp_field(res, target, field_deg=FIELD_DEG, pa_deg=PA_DEG)
plt.show()

## Schritt 5 — Ergebnis speichern

Schreibt `field_config.json` (Eingabe für Notebook 04): Ziel, Comps/Check/Reserve mit AUID, Position, V/B und Fehler, dazu alle
verwendeten Parameter und die Paketversion — damit ist die Auswahl später nachvollziehbar.

In [ ]:
sel = res[[str(r) in ("comp", "check", "reserve") for r in res["role"]]]
cfg_used = {**VSP_CFG, **CFG_FELD}
record = {
    "erzeugt": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "varphot": __version__,
    "ziel": target,
    "vsp": {"stern": VSP_STAR, "fov_arcmin": FOV_ARCMIN, "maglimit": MAGLIMIT},
    "regeln": {k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg_used.items()},
    "sterne": [
        {"rolle": str(r["role"]), "auid": str(r["auid"]), "label": str(r["label"]),
         "ra": float(r["ra"]), "dec": float(r["dec"]), "V": float(r["V"]), "V_err": float(r["V_err"]),
         "B": float(r["B"]), "B_err": float(r["B_err"]), "G": None if r["G"] != r["G"] else float(r["G"])}
        for r in sel
    ],
}
OUTPUT.write_text(json.dumps(record, ensure_ascii=False, indent=1))
print(f"gespeichert: {OUTPUT}  ({len(record['sterne'])} Sterne: "
      f"{sum(s['rolle']=='comp' for s in record['sterne'])} Comps, "
      f"{sum(s['rolle']=='check' for s in record['sterne'])} Check, "
      f"{sum(s['rolle']=='reserve' for s in record['sterne'])} Reserve)")

## Ausblick — Zuordnung im Bild

Sobald ein Bild des Zielfeldes vorliegt (WCS aus Notebook 02, Quellenliste `<bild>.sources.ecsv`), werden die Sterne aus
`field_config.json` per Himmelskoordinaten den erkannten Quellen zugeordnet (`crossmatch`), auf Sättigung, Bildrand und Nachbarn im Bild
geprüft (`select_comps`) und in der Sternkarte markiert (`chart_map`). Das gehört zu Roadmap P0-3.